# M08 lab solution: first graph

**Brief.** Build a graph that classifies a question, routes it to one of two nodes, and returns a structured answer.

A `StateGraph` classifies each Larkspur question as billing, technical or general with structured output, then a conditional edge sends billing questions to `billing_answer` (which searches only the billing, refunds and SLA files) and the rest to `support_answer` (the whole handbook). Each answer node returns a Pydantic `Answer` with sources. The solution draws the graph and shows the route for 10 golden questions.

Key: `EXPERIENTIALLABS_API_KEY`.

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

sys.path.insert(0, str(Path("../../../../src").resolve()))
load_dotenv()

DATA = Path("../data")

from typing import Literal, TypedDict

import pandas as pd
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

from helpers.utils import get_experientiallabs_llm

In [ ]:
from lab_kit import load_handbook, load_jsonl, tfidf_retriever
from langchain_text_splitters import RecursiveCharacterTextSplitter

pages = load_handbook(DATA / "larkspur" / "handbook")
golden = load_jsonl(DATA / "larkspur" / "golden.jsonl")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [c for p in pages for c in splitter.split_text(p["text"])]
print(len(pages), "handbook files,", len(chunks), "chunks,", len(golden), "golden questions")

## 1. State, nodes and the router

Each node returns only the keys it changes. The router reads `category` and returns a node name; passing the destination list to `add_conditional_edges` makes an unknown route fail loudly and puts both branches in the drawing.

In [ ]:
class Route(BaseModel):
    category: Literal["billing", "technical", "general"]

class Answer(BaseModel):
    answer: str = Field(description="The answer, using only the passages")
    sources: list[str] = Field(description="Handbook files the answer uses, such as 02_refunds.md")

class State(TypedDict, total=False):
    question: str
    category: str
    node: str
    answer: dict

BILLING_FILES = {"01_billing.md", "02_refunds.md", "12_sla_credits.md"}

def passages_from(sources: set[str] | None) -> list[tuple[str, str]]:
    return [(p["source"], c) for p in pages if sources is None or p["source"] in sources
            for c in splitter.split_text(p["text"])]

billing_passages = passages_from(BILLING_FILES)
all_passages = passages_from(None)
billing_search = tfidf_retriever([c for _, c in billing_passages], k=3)
support_search = tfidf_retriever([c for _, c in all_passages], k=3)
source_of = {c: s for s, c in all_passages}

llm = get_experientiallabs_llm()
classifier = llm.with_structured_output(Route)
answerer = llm.with_structured_output(Answer)

def classify(state: State) -> dict:
    route = classifier.invoke(
        "Classify this customer question about a cloud hosting company as billing (payments, invoices, "
        "refunds, credits), technical (servers, API, backups, regions, security) or general (anything else).\n\n"
        f"Question: {state['question']}")
    return {"category": route.category}

def route(state: State) -> str:
    return "billing_answer" if state["category"] == "billing" else "support_answer"

def answer_from(state: State, found: list[str], node: str) -> dict:
    context = "\n\n".join(f"[{source_of[c]}] {c}" for c in found)
    result = answerer.invoke(
        "Answer the customer's question using only these passages from the Larkspur handbook. "
        "List the file names you used as sources. If the passages do not answer it, say so.\n\n"
        f"{context}\n\nQuestion: {state['question']}")
    return {"node": node, "answer": result.model_dump()}

def billing_answer(state: State) -> dict:
    return answer_from(state, billing_search(state["question"]), "billing_answer")

def support_answer(state: State) -> dict:
    return answer_from(state, support_search(state["question"]), "support_answer")

builder = StateGraph(State)
builder.add_node("classify", classify)
builder.add_node("billing_answer", billing_answer)
builder.add_node("support_answer", support_answer)
builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route, ["billing_answer", "support_answer"])
builder.add_edge("billing_answer", END)
builder.add_edge("support_answer", END)

graph = builder.compile()

## 2. The graph, drawn

In [ ]:
print(graph.get_graph().draw_mermaid())

## 3. Route 10 golden questions

In [ ]:
questions = [g for g in golden if g["evidence_spans"]][:10]
rows = []
for g in questions:
    out = graph.invoke({"question": g["question"]})
    rows.append({"id": g["id"], "question": g["question"], "category": out["category"], "node": out["node"],
                 "answer": out["answer"]["answer"], "sources": out["answer"]["sources"]})
runs = pd.DataFrame(rows)

## Results

In [ ]:
pd.set_option("display.max_colwidth", 160)
runs

In [ ]:
runs.groupby(["category", "node"]).size().rename("questions").reset_index()

## What this shows

- The route is a plain function of state, and the classifier's `Literal` type means it can only return one of the three labels.
- The drawing shows both branches because the destinations were listed; without the list the run would still work but the graph would draw no branches.
- Reading the category column against the questions shows where the classifier and your own reading disagree, which is the input a better prompt or a fallback route needs.